In [ ]:
import numpy as np
from random import random
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../src')
import data_import as di
import pickle
import os
import pandas as pd
%matplotlib inline

In [ ]:
def data_arrange(session,transition_type,sessionID):
# transition_type:
# A: common transitions are left(1) -> up(1) and right(0) -> down(0)
# B: common transitions are left(1)->down(0) and right(0) -> up(1)

    b = session.print_lines
    n_trial = len(b)
    subject = np.zeros(n_trial) + session.subject_ID
    session = np.zeros(n_trial) + sessionID
    choice = np.zeros(n_trial) # 0 = right, 1 = left
    second_state = np.zeros(n_trial) # 0 = down, 1 = Up
    outcome = np.zeros(n_trial) # 0 = unrewarded, 1 = rewarded 
    transition = np.zeros(n_trial) # 0 = rare, 1 = common
    trial_type = np.zeros(n_trial) # 0 = forced choice trial, 1 = free choice trial
    reward_block = np.zeros(n_trial) # 0 = down rewarding, 1= up rewarding, 2 = neutral
    correct    = np.zeros(n_trial)

    for i in range(len(b)): 
        data = b[i].split(" ")
        if data[4] == 'C:1':
            choice[i] = 1
        if data[5] == 'S:1':
            second_state[i] = 1
        if data[6] == 'O:1':
            outcome[i] = 1
        if data[8][2] == 'D':
            reward_block[i] = 0
        elif data[8][2] == 'U':
            reward_block[i] = 1
        else:
            reward_block[i] = 2
        if data[9] == 'CT:FC':
            trial_type[i] = 1
        if transition_type == 'A':
            transition[i] = 1 if choice[i] == second_state[i] else 0
            if reward_block[i] == 0 or 1:
                correct[i]    = 1 if choice[i] == reward_block[i] else 0
            else:
                correct[i] = 2
        else:
            transition[i] = 0 if choice[i] == second_state[i] else 1
            if reward_block[i] == 1 or 2:
                correct[i]    = 0 if choice[i] == reward_block[i] else 1
            else:
                correct[i] = 2
    
    return choice, transition, second_state, outcome, correct, trial_type, reward_block, session, subject

In [ ]:
def exportdata(session,folder,file_name,file_name2):
    stay = session['choice'][1:] == session['choice'][:-1]

    d = {'stay':stay.astype(int),
         'choice':session['choice'][:-1],
         'transition':session['transition'][:-1],
         'secondstate':session['proximal'][:-1],
         'outcome':session['outcome'][:-1].astype(int),
         'correct':session['correct'][:-1].astype(int),
         'trial type':session['trial type'][:-1],
         'current trial type':session['trial type'][1:], 
         'block'  :session['block'][:-1],
         'session':session['session'][:-1],
         'subject':session['subjectID'][:-1]
        }

    df = pd.DataFrame(data = d)
    df2 = pd.DataFrame(data=session)

    if not os.path.exists(folder):  
        os.mkdir(folder)  
        
    fullname = os.path.join(folder, file_name)
    fullname2 = os.path.join(folder, file_name2)
    df.to_csv(fullname,mode='a', index=False, header=False)
    df2.to_csv(fullname2,mode='a', index=False, header=False)

In [ ]:
#Experimental Data

data_dir = '/Users/maoyasueda/Documents/OIST/WT_data/'

WT1experiment = di.Experiment(data_dir+'WT1')
WT1experiment.save()
WT1 = WT1experiment.get_sessions(subject_IDs='all')

WT2experiment = di.Experiment(data_dir+'WT2')
WT2experiment.save()
WT2 = WT2experiment.get_sessions(subject_IDs='all')

WT3experiment = di.Experiment(data_dir+'WT3')
WT3experiment.save()
WT3 = WT3experiment.get_sessions(subject_IDs='all')

WT4experiment = di.Experiment(data_dir+'WT4')
WT4experiment.save()
WT4 = WT4experiment.get_sessions(subject_IDs='all')

WT5experiment = di.Experiment(data_dir+'WT5')
WT5experiment.save()
WT5 = WT5experiment.get_sessions(subject_IDs='all')

WT6experiment = di.Experiment(data_dir+'WT6')
WT6experiment.save()
WT6 = WT6experiment.get_sessions(subject_IDs='all')


In [ ]:
Stage4_8 = WT1+WT2+WT3+WT4+WT5+WT6
n_session = len(Stage4_8)
s = np.concatenate((np.arange(len(WT1))+1,np.arange(len(WT2))+1,np.arange(len(WT3))+1,np.arange(len(WT4))+1,
    np.arange(len(WT5))+1, np.arange(len(WT6))+1))
s
trans_type = [['A'] * len(WT1),['B'] * len(WT2),['A'] * len(WT3),['B'] * len(WT4),['A'] * len(WT5),['B'] * len(WT6)]
trans_type = [item for sublist in trans_type for item in sublist]

In [ ]:
print(Stage4_8)

In [ ]:
data = Stage4_8
n_session = len(data)

folder = './LogisticRegression'
file_name = './Stage4.8_stay.csv'
file_name2 = './Stage4.8_sessions.csv'

for i in range(n_session):
    choices, transitions,ss, outcomes, correct, tt, blocks, sessionID, subject = data_arrange(data[i],trans_type[i],s[i])
    session = {'choice':choices,
          'transition':transitions,
          'proximal':ss,
          'outcome':outcomes,
          'block':blocks,
          'correct':correct,
          'trial type':tt,
          'session': sessionID,
          'subjectID':subject}
    
    exportdata(session,folder,file_name,file_name2) 

In [ ]:
#Read rowdata
rowdata = {}
for i in range(1,7):
    training_folder = di.Experiment(data_dir +'WT' + str(i) +  '_training')
    tr_data = training_folder.get_sessions(subject_IDs='all', when='all')
    
    
    name = "WT_tr" + str(i)
    
    rowdata[name] = tr_data

In [ ]:
print(rowdata)

In [ ]:
trainig4_7 = [
    session 
    for sessions in rowdata.values()    # Iterate over all lists in the dictionary
    for session in sessions             # Iterate over each session within a list
    if session.training_stage == "4.7"  # Filter condition for training stage
]

training4_6 = [
    session 
    for sessions in rowdata.values() 
    for session in sessions 
    if session.training_stage == "4.6"
]

In [ ]:
data = trainig4_7
n_session = len(data)

folder = './LogisticRegression'
file_name = './Stage4.7_stay.csv'
file_name2 = './Stage4.7_sessions.csv'

for i in range(n_session):
    choices, transitions,ss, outcomes, correct, tt, blocks, sessionID, subject = data_arrange(data[i],trans_type[i],s[i])
    session = {'choice':choices,
          'transition':transitions,
          'proximal':ss,
          'outcome':outcomes,
          'block':blocks,
          'correct':correct,
          'trial type':tt,
          'session': sessionID,
          'subjectID':subject}
    
    exportdata(session,folder,file_name,file_name2) 


data = training4_6
n_session = len(data)

folder = './LogisticRegression'
file_name = './Stage4.6_stay.csv'
file_name2 = './Stage4.6_sessions.csv'

for i in range(n_session):
    choices, transitions,ss, outcomes, correct, tt, blocks, sessionID, subject = data_arrange(data[i],trans_type[i],s[i])
    session = {'choice':choices,
          'transition':transitions,
          'proximal':ss,
          'outcome':outcomes,
          'block':blocks,
          'correct':correct,
          'trial type':tt,
          'session': sessionID,
          'subjectID':subject}
    
    exportdata(session,folder,file_name,file_name2) 

In [ ]:
i = 0
choices, transitions,ss, outcomes, correct, tt, blocks, sessionID, subject = data_arrange(data[i],trans_type[i],s[i])

In [ ]:
session['choice'][:-1]

In [ ]:
stay = session['choice'][1:] == session['choice'][:-1]
stay.astype(int)

## Stay probability — stages 4.6, 4.7, 4.8

Computes per-subject stay probability for 4 conditions
(common-rewarded, rare-rewarded, common-unrewarded, rare-unrewarded)
using adjacent FC→FC trial pairs only. Saves a per-subject CSV and
produces a bar plot per stage (mean ± SEM across subjects, with
individual subjects overlaid).

In [ ]:
# Setup: config, imports, session loader
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_here = os.path.abspath('')
_root = os.path.dirname(_here) if os.path.basename(_here) == 'notebooks' else _here
sys.path.insert(0, _root)
sys.path.insert(0, os.path.join(_root, 'src'))

from config import (
    SUBJECTS, STAGES, raw_subject_dir, FIGURES_DIR, PROJECT_ROOT
)
from src.data_import import Experiment

STAY_STAGES = ['4.6', '4.7', '4.8']
BEHAVIOR_DIR = os.path.join(PROJECT_ROOT, 'results', 'behavior')
os.makedirs(BEHAVIOR_DIR, exist_ok=True)

print('Subjects:', SUBJECTS)
print('Stages  :', STAY_STAGES)
print('Output  :', BEHAVIOR_DIR)

In [ ]:
# Load sessions per (subject, stage)
# Stage 4.8: all sessions in the main folder (no filter).
# Stages 4.6/4.7: only _Training folder, filtered by training_stage.

def load_sessions(subject, stage):
    if stage == '4.8':
        folder = raw_subject_dir(subject, training=False)
        if not os.path.isdir(folder):
            return []
        sessions = list(Experiment(folder).sessions)
    else:
        folder = raw_subject_dir(subject, training=True)
        if not os.path.isdir(folder):
            return []
        sessions = [s for s in Experiment(folder).sessions
                    if s.training_stage == stage]
    seen, unique = set(), []
    for s in sorted(sessions, key=lambda s: s.datetime_string):
        if s.file_name not in seen:
            seen.add(s.file_name)
            unique.append(s)
    return unique

In [ ]:
# Parse one session into the arrays needed for stay probability
# Conventions (match 6_behavioral_simulation.ipynb Cell 4):
#   c  : 1=left, 2=right          ss : 1=up, 2=down
#   tt : 1=free choice, 2=forced  r  : 1.0=rewarded, 0.0=not
#   trans_type: 'A' or 'B' (from d[8][3] of the first trial)

def parse_session(session):
    b = session.print_lines
    T = len(b)
    trans_type = b[0].split(' ')[8][3]  # 'A' or 'B'
    c  = np.ones(T, dtype=int)
    ss = np.ones(T, dtype=int)
    tt = np.ones(T, dtype=int)
    r  = np.zeros(T)
    for j in range(T):
        d = b[j].split(' ')
        if d[4] == 'C:0': c[j]  = 2
        if d[5] == 'S:0': ss[j] = 2
        r[j] = 1.0 if d[6] == 'O:1' else 0.0
        if d[9] != 'CT:FC': tt[j] = 2
    return c, ss, tt, r, trans_type


def stay_prob_4cond(c, ss, tt, r, trans_type):
    """Return [rew_com, rew_rare, unrew_com, unrew_rare], NaN if no data.

    Only adjacent FC→FC trial pairs are counted.
    Type A: matching c==ss → common.  Type B: c!=ss → common.
    """
    stay  = np.zeros(4)
    total = np.zeros(4)
    for t in range(len(c) - 1):
        if tt[t] != 1 or tt[t + 1] != 1:
            continue
        if trans_type == 'A':
            is_common = (c[t] == ss[t])
        else:
            is_common = (c[t] != ss[t])
        rew_idx  = 0 if r[t] == 1.0 else 1
        comm_idx = 0 if is_common else 1
        cond     = rew_idx * 2 + comm_idx
        stay[cond]  += int(c[t + 1] == c[t])
        total[cond] += 1
    return np.where(total > 0, stay / total, np.nan)

In [ ]:
# Compute per-session stay probability -> save long-format CSV
COND_LABELS = ['rew_com', 'rew_rare', 'unrew_com', 'unrew_rare']

rows = []
for stage in STAY_STAGES:
    for subject in SUBJECTS:
        for sess_idx, session in enumerate(load_sessions(subject, stage)):
            c, ss, tt, r, ttype = parse_session(session)
            sp = stay_prob_4cond(c, ss, tt, r, ttype)
            rows.append({
                'subject':     subject,
                'stage':       stage,
                'session_idx': sess_idx,
                'date':        session.datetime_string,
                'trans_type':  ttype,
                'n_trials':    len(c),
                **{f'stay_{lab}': sp[i] for i, lab in enumerate(COND_LABELS)},
            })

df_sess = pd.DataFrame(rows)
sess_csv = os.path.join(BEHAVIOR_DIR, 'stay_prob_per_session.csv')
df_sess.to_csv(sess_csv, index=False)
print(f'Saved {len(df_sess)} session rows to {sess_csv}')
df_sess.head()

In [ ]:
# Collapse to per-subject (average across sessions) -> save CSV
stay_cols = [f'stay_{lab}' for lab in COND_LABELS]

df_subj = (df_sess.groupby(['stage', 'subject'])[stay_cols]
                  .mean()
                  .reset_index())
subj_csv = os.path.join(BEHAVIOR_DIR, 'stay_prob_per_subject.csv')
df_subj.to_csv(subj_csv, index=False)
print(f'Saved {len(df_subj)} subject-stage rows to {subj_csv}')
df_subj

In [ ]:
# Plot: one panel per stage, 4 bars (C1, R1, C0, R0)
#   Bars   = mean across subjects, error bars = SEM
#   Dots   = individual subjects (jittered)
#   Colors : green=common, orange=rare; lighter alpha=unrewarded

COND_DISPLAY = ['C1', 'R1', 'C0', 'R0']
COND_COLORS  = ['#10A551', '#F6921E', '#10A551', '#F6921E']
COND_ALPHA   = [1.0, 1.0, 0.5, 0.5]

fig, axes = plt.subplots(1, len(STAY_STAGES),
                         figsize=(3.2 * len(STAY_STAGES), 3.5),
                         sharey=True, dpi=150)
rng = np.random.default_rng(0)
x = np.arange(4)

for ax, stage in zip(axes, STAY_STAGES):
    subj = df_subj[df_subj['stage'] == stage].set_index('subject')[stay_cols]
    m = subj.mean(axis=0).values
    sem = subj.sem(axis=0).values

    for xi in x:
        ax.bar(xi, m[xi], yerr=sem[xi],
               color=COND_COLORS[xi], alpha=COND_ALPHA[xi],
               capsize=4, width=0.7,
               error_kw={'linewidth': 1.5})

    for _, row_vals in subj.iterrows():
        jitter = rng.uniform(-0.08, 0.08, size=4)
        ax.plot(x + jitter, row_vals.values,
                'o', color='k', ms=4, alpha=0.6, zorder=5)

    ax.set_xticks(x)
    ax.set_xticklabels(COND_DISPLAY)
    ax.set_title(f'Stage {stage}  (n={subj.shape[0]})')
    ax.set_ylim(0, 1)
    ax.axhline(0.5, color='gray', lw=0.6, ls='--', alpha=0.5)

axes[0].set_ylabel('Stay probability')
plt.tight_layout()

fig_path = os.path.join(FIGURES_DIR, 'stay_prob_experimental_stages.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')